# VibeCheck Entertainment: Talent Performance Mapping

This notebook transforms a public music-features dataset into a Tableau-ready file for the
"Atmosphere Talent Performance Index" project.

The goal is to create business-style fields such as:
- Talent Category
- Venue Type Fit
- Time Slot
- Vibe Impact Score
- Success Probability
- Budget Range
- Guest Count Band
- ROI Engagement Score

In [ ]:
import pandas as pd
import numpy as np

## Load the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving dataset.csv to dataset.csv


In [ ]:
df = pd.read_csv("dataset.csv")
df.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## Keep only the columns needed for the project

In [ ]:
columns_needed = [
    "artists",
    "track_name",
    "track_genre",
    "popularity",
    "danceability",
    "energy",
    "valence",
    "tempo",
    "liveness",
    "acousticness",
    "speechiness",
    "instrumentalness"
]

df = df[columns_needed].copy()
df.head()

,artists,track_name,track_genre,popularity,danceability,energy,valence,tempo,liveness,acousticness,speechiness,instrumentalness
0,Gen Hoshino,Comedy,acoustic,73,0.676,0.4610,0.715,87.917,0.3580,0.0322,0.1430,0.000001
1,Ben Woodward,Ghost - Acoustic,acoustic,55,0.420,0.1660,0.267,77.489,0.1010,0.9240,0.0763,0.000006
2,Ingrid Michaelson;ZAYN,To Begin Again,acoustic,57,0.438,0.3590,0.120,76.332,0.1170,0.2100,0.0557,0.000000
3,Kina Grannis,Can't Help Falling In Love,acoustic,71,0.266,0.0596,0.143,181.740,0.1320,0.9050,0.0363,0.000071
4,Chord Overstreet,Hold On,acoustic,82,0.618,0.4430,0.167,119.949,0.0829,0.4690,0.0526,0.000000


## Drop rows with missing values in the main scoring columns

In [ ]:
score_cols = [
    "popularity",
    "danceability",
    "energy",
    "valence",
    "tempo",
    "liveness",
    "acousticness",
    "speechiness",
    "instrumentalness"
]

df = df.dropna(subset=score_cols).copy()
df.shape

(114000, 12)

## Create Talent Category

This maps track genres into business-friendly entertainment categories.

In [ ]:
def map_talent_category(genre):
    g = str(genre).lower()

    if any(x in g for x in ["jazz", "blues", "soul", "swing"]):
        return "Jazz / Lounge"
    elif any(x in g for x in ["acoustic", "folk", "singer-songwriter", "classical"]):
        return "Acoustic / Intimate"
    elif any(x in g for x in ["edm", "house", "dance", "techno", "trance", "club"]):
        return "DJ / Club Set"
    elif any(x in g for x in ["hip hop", "rap", "trap", "drill", "afrobeats"]):
        return "Hype / Urban Set"
    elif any(x in g for x in ["ambient", "chill", "lo-fi", "new-age"]):
        return "Ambient / Cocktail"
    elif any(x in g for x in ["rock", "metal", "punk", "alternative"]):
        return "High-Energy Live Act"
    elif any(x in g for x in ["latin", "reggaeton", "salsa", "funk"]):
        return "Rhythm / Dance Ensemble"
    else:
        return "Versatile Entertainment"

df["Talent_Category"] = df["track_genre"].apply(map_talent_category)
df["Talent_Category"].value_counts().head(10)

,count
Talent_Category,
Versatile Entertainment,71000
High-Energy Live Act,15000
DJ / Club Set,13000
Rhythm / Dance Ensemble,5000
Acoustic / Intimate,4000
Jazz / Lounge,3000
Ambient / Cocktail,3000


## Create Venue Type Fit

This uses sound and mood features to estimate the venue type where the act fits best.

In [ ]:
def map_venue_type(row):
    energy = row["energy"]
    dance = row["danceability"]
    acoustic = row["acousticness"]
    live = row["liveness"]
    valence = row["valence"]

    if acoustic >= 0.6 and energy <= 0.5:
        return "Rooftop Lounge"
    elif dance >= 0.7 and energy >= 0.7:
        return "Grand Ballroom"
    elif valence >= 0.6 and energy >= 0.5 and dance >= 0.5:
        return "Cocktail Reception"
    elif live >= 0.5 and energy >= 0.6:
        return "Outdoor Festival Space"
    elif acoustic >= 0.4 and valence >= 0.5:
        return "Private Dining Venue"
    else:
        return "Flexible Event Space"

df["Venue_Type_Fit"] = df.apply(map_venue_type, axis=1)
df["Venue_Type_Fit"].value_counts()

,count
Venue_Type_Fit,
Flexible Event Space,51286
Rooftop Lounge,21181
Cocktail Reception,20768
Grand Ballroom,12641
Outdoor Festival Space,4664
Private Dining Venue,3460


## Create Time Slot

This estimates the event time slot where the act is most suitable.

In [ ]:
def map_time_slot(row):
    energy = row["energy"]
    dance = row["danceability"]
    acoustic = row["acousticness"]
    valence = row["valence"]

    if acoustic >= 0.6 and energy <= 0.4:
        return "Dinner Set"
    elif energy <= 0.5 and valence >= 0.5:
        return "Cocktail Hour"
    elif dance >= 0.7 and energy >= 0.7:
        return "Late Night"
    elif energy >= 0.5 and valence >= 0.5:
        return "Main Event"
    else:
        return "Flexible Slot"

df["Time_Slot"] = df.apply(map_time_slot, axis=1)
df["Time_Slot"].value_counts()

,count
Time_Slot,
Flexible Slot,45376
Main Event,33717
Dinner Set,16960
Late Night,12641
Cocktail Hour,5306


## Create Vibe Impact Score

This score reflects how strongly the act may shape the emotional and energy tone of the event.

In [ ]:
df["Vibe_Impact_Score"] = (
    df["energy"] * 0.30 +
    df["danceability"] * 0.25 +
    df["valence"] * 0.20 +
    (df["popularity"] / 100) * 0.25
) * 100

df["Vibe_Impact_Score"] = df["Vibe_Impact_Score"].round(2)
df["Vibe_Impact_Score"].describe()

,Vibe_Impact_Score
count,114000.000000
mean,51.202451
std,13.555047
min,0.940000
25%,43.510000
50%,52.820000
75%,60.780000
max,91.720000



## Create Success Probability

This is a synthetic business score to estimate likely audience response and planner satisfaction.

In [ ]:
df["Success_Probability"] = (
    (df["popularity"] / 100) * 0.35 +
    df["danceability"] * 0.20 +
    df["valence"] * 0.20 +
    df["liveness"] * 0.15 +
    df["energy"] * 0.10
) * 100

df["Success_Probability"] = df["Success_Probability"].round(2)
df["Success_Probability"].describe()

,Success_Probability
count,114000.000000
mean,42.068176
std,11.819128
min,2.120000
25%,34.380000
50%,42.320000
75%,50.370000
max,82.980000


## Create Budget Range

This is a simple synthetic budget band based on popularity and production intensity.

In [ ]:
def map_budget_range(row):
    pop = row["popularity"]
    energy = row["energy"]
    live = row["liveness"]

    score = (pop / 100) + energy + live

    if score < 1.0:
        return "Low"
    elif score < 1.8:
        return "Medium"
    else:
        return "High"

df["Budget_Range"] = df.apply(map_budget_range, axis=1)
df["Budget_Range"].value_counts()

,count
Budget_Range,
Medium,69603
Low,36874
High,7523


## Create Guest Count Band

This estimates the event size the act is best suited for.

In [ ]:
def map_guest_count_band(row):
    energy = row["energy"]
    live = row["liveness"]
    dance = row["danceability"]

    score = energy + live + dance

    if score < 1.2:
        return "Small (0-100)"
    elif score < 2.0:
        return "Medium (101-250)"
    else:
        return "Large (251+)"

df["Guest_Count_Band"] = df.apply(map_guest_count_band, axis=1)
df["Guest_Count_Band"].value_counts()

,count
Guest_Count_Band,
Medium (101-250),79716
Small (0-100),28319
Large (251+),5965


## Create ROI Engagement Score

This gives a synthetic measure of how much perceived event value the act may return for the planner.

In [ ]:
budget_map = {"Low": 1, "Medium": 2, "High": 3}
guest_map = {"Small (0-100)": 1, "Medium (101-250)": 2, "Large (251+)": 3}

df["Budget_Num"] = df["Budget_Range"].map(budget_map)
df["Guest_Num"] = df["Guest_Count_Band"].map(guest_map)

df["ROI_Engagement_Score"] = (
    df["Success_Probability"] * 0.5 +
    df["Vibe_Impact_Score"] * 0.3 +
    (4 - df["Budget_Num"]) * 5 +
    df["Guest_Num"] * 5
)

df["ROI_Engagement_Score"] = df["ROI_Engagement_Score"].round(2)
df["ROI_Engagement_Score"].describe()

,ROI_Engagement_Score
count,114000.000000
mean,56.701711
std,9.804082
min,21.410000
25%,50.690000
50%,57.710000
75%,63.610000
max,89.010000


## Keep only the final project-ready columns

In [ ]:
final_columns = [
    "artists",
    "track_name",
    "track_genre",
    "popularity",
    "danceability",
    "energy",
    "valence",
    "tempo",
    "liveness",
    "acousticness",
    "speechiness",
    "instrumentalness",
    "Talent_Category",
    "Venue_Type_Fit",
    "Time_Slot",
    "Vibe_Impact_Score",
    "Success_Probability",
    "Budget_Range",
    "Guest_Count_Band",
    "ROI_Engagement_Score"
]

df_final = df[final_columns].copy()
df_final.head()

,artists,track_name,track_genre,popularity,danceability,energy,valence,tempo,liveness,acousticness,speechiness,instrumentalness,Talent_Category,Venue_Type_Fit,Time_Slot,Vibe_Impact_Score,Success_Probability,Budget_Range,Guest_Count_Band,ROI_Engagement_Score
0,Gen Hoshino,Comedy,acoustic,73,0.676,0.4610,0.715,87.917,0.3580,0.0322,0.1430,0.000001,Acoustic / Intimate,Flexible Event Space,Cocktail Hour,63.28,63.35,Medium,Medium (101-250),70.66
1,Ben Woodward,Ghost - Acoustic,acoustic,55,0.420,0.1660,0.267,77.489,0.1010,0.9240,0.0763,0.000006,Acoustic / Intimate,Rooftop Lounge,Dinner Set,34.57,36.17,Low,Small (0-100),48.46
2,Ingrid Michaelson;ZAYN,To Begin Again,acoustic,57,0.438,0.3590,0.120,76.332,0.1170,0.2100,0.0557,0.000000,Acoustic / Intimate,Flexible Event Space,Flexible Slot,38.37,36.46,Medium,Small (0-100),44.74
3,Kina Grannis,Can't Help Falling In Love,acoustic,71,0.266,0.0596,0.143,181.740,0.1320,0.9050,0.0363,0.000071,Acoustic / Intimate,Rooftop Lounge,Dinner Set,29.05,35.61,Low,Small (0-100),46.52
4,Chord Overstreet,Hold On,acoustic,82,0.618,0.4430,0.167,119.949,0.0829,0.4690,0.0526,0.000000,Acoustic / Intimate,Flexible Event Space,Flexible Slot,52.58,50.07,Medium,Small (0-100),55.81


In [ ]:
print(df_final.shape)
print(df_final.columns.tolist())

(114000, 20)
['artists', 'track_name', 'track_genre', 'popularity', 'danceability', 'energy', 'valence', 'tempo', 'liveness', 'acousticness', 'speechiness', 'instrumentalness', 'Talent_Category', 'Venue_Type_Fit', 'Time_Slot', 'Vibe_Impact_Score', 'Success_Probability', 'Budget_Range', 'Guest_Count_Band', 'ROI_Engagement_Score']


## Save the transformed dataset as CSV

In [ ]:
df_final.to_csv("vibecheck_talent_index_ready.csv", index=False)
print("Saved as vibecheck_talent_index_ready.csv")

Saved as vibecheck_talent_index_ready.csv


In [ ]:
from google.colab import files
files.download("vibecheck_talent_index_ready.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>